# Averaged Waveforms by Cell Type

This notebook loads the processed waveforms dictionary and cell metadata, inverts Excitatory signals if needed, and computes averaged waveforms grouped by cell type. The workflow is similar to the refactored averaged waveforms notebook, but grouping is by cell type instead of frequency/speed. Visualizations and CSV export are included.

## 1 · Import Required Libraries

In [ ]:
import sys, os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ── make sure the scripts package is importable ──────────────────────────────
REPO_ROOT = Path(os.getcwd()).parents[1]
sys.path.insert(0, str(REPO_ROOT / "analysis" / "scripts"))

from waveforms import bin_wave

## 2 · Load Data

- Loads `all_waveforms.pkl` (from main pipeline)
- Loads `summary_spikes2.csv` for cell type info

In [ ]:
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
WAVEFORMS_NESTED_PKL = PROCESSED_DIR / "all_waveforms_nested.pkl"
WAVEFORMS_PKL = PROCESSED_DIR / "all_waveforms.pkl"
METADATA_CSV = REPO_ROOT / "data" / "annotations" / "summary_spikes2.csv"

if WAVEFORMS_NESTED_PKL.exists():
    with open(WAVEFORMS_NESTED_PKL, "rb") as f:
        all_waveforms = pickle.load(f)
    print("Loaded nested waveform dictionary from all_waveforms_nested.pkl")
else:
    with open(WAVEFORMS_PKL, "rb") as f:
        all_waveforms = pickle.load(f)
    print("Loaded waveform dictionary from all_waveforms.pkl")

cell_types_df = pd.read_csv(METADATA_CSV)
cell_types_df = cell_types_df.set_index("Cell")

print(f"Loaded waveform object with top-level length {len(all_waveforms)}.")
print(f"Loaded cell type metadata for {len(cell_types_df)} cells.")

## 3 · Helper Functions

- `get_cell_type()` — maps cell name to cell type
- `invert_excitatory()` — inverts Excitatory signals if not already inverted
- `normalize_signal_type()` — standardizes signal type labels

In [ ]:
def get_cell_type(cell_name):
    """Return cell type for a given cell name."""
    try:
        return cell_types_df.loc[cell_name, "Cell Type"]
    except Exception:
        return None


def normalize_signal_type(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.lower().startswith("excit"): return "Excitatory"
    if s.lower().startswith("inhib"): return "Inhibitory"
    return None


def invert_excitatory(wave_df, signal_type):
    """
    Invert Excitatory signals (multiply Current and Normalized Current by -1)
    if not already inverted (assumes positive-going is not inverted).
    """
    if signal_type != "Excitatory":
        return wave_df
    # Heuristic: if mean(Normalized Current) > 0.5, invert
    if "Normalized Current" in wave_df.columns:
        if wave_df["Normalized Current"].mean() > 0.5:
            wave_df = wave_df.copy()
            wave_df["Current"] = -wave_df["Current"]
            wave_df["Normalized Current"] = 1 - wave_df["Normalized Current"]
    return wave_df

## 4 · Build Averaged Waveforms by Cell Type

- For each waveform, assign cell type and normalized signal type
- Invert Excitatory signals if needed
- Bin each waveform (default 100 bins)
- Group by (cell type, signal type, phase bin) and average

In [ ]:
NUM_BINS = 100

is_flat_dict = bool(all_waveforms) and isinstance(next(iter(all_waveforms.keys())), tuple)
if is_flat_dict:
    raise ValueError(
        "Cell-type averaging requires the nested waveform dictionary. "
        "Re-run main_pipeline.ipynb so it saves all_waveforms_nested.pkl."
    )

all_rows = []
for cell_name, traces in all_waveforms.items():
    cell_type = get_cell_type(cell_name)
    if cell_type is None:
        continue
    for trace_name, waveforms in traces.items():
        for key, wave_df in waveforms.items():
            signal_type = normalize_signal_type(key[1])
            if signal_type is None:
                continue
            wave_df = invert_excitatory(wave_df, signal_type)
            if wave_df.empty or "Phase" not in wave_df.columns:
                continue
            binned = bin_wave(wave_df.copy(), num_bins=NUM_BINS)
            binned["cell type"] = cell_type
            binned["signal type"] = signal_type
            all_rows.append(binned)

if not all_rows:
    print("⚠️  No rows collected — check your data or column names.")
    averaged_df = pd.DataFrame()
else:
    rows_df = pd.concat(all_rows, ignore_index=True)
    averaged_df = (
        rows_df
        .groupby(["cell type", "signal type", "Phase"], observed=True)["Normalized Current"]
        .mean()
        .reset_index()
    )
    print(f"Total binned rows  : {len(rows_df):,}")
    print(f"Averaged rows      : {len(averaged_df):,}")
    print(f"Groups             : {averaged_df[['cell type','signal type']].drop_duplicates().shape[0]}")

averaged_df.head(10)

## 5 · Export to CSV

In [ ]:
if not averaged_df.empty:
    out_fname = "averaged_waveforms_by_celltype.csv"
    out_path  = PROCESSED_DIR / out_fname
    averaged_df.to_csv(out_path, index=False)
    print(f"Saved → {out_path}")
else:
    print("Nothing to save.")

## 6 · Visualize Averaged Waveforms by Cell Type

Each subplot is a cell type; Excitatory and Inhibitory are overlaid.

In [ ]:
def plot_by_celltype(averaged_df, signal_colors=None, fig_title=None):
    if averaged_df.empty:
        print("No data to plot.")
        return
    if signal_colors is None:
        signal_colors = {"Excitatory": "#d62728", "Inhibitory": "#1f77b4"}
    cell_types = sorted(averaged_df["cell type"].unique())
    n = len(cell_types)
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3*nrows), sharex=True, sharey=True)
    axes = axes.flat if n > 1 else [axes]
    for i, cell_type in enumerate(cell_types):
        ax = axes[i]
        subset = averaged_df[averaged_df["cell type"] == cell_type]
        for sig_type, color in signal_colors.items():
            sig_data = subset[subset["signal type"] == sig_type]
            if not sig_data.empty:
                sig_sorted = sig_data.sort_values("Phase")
                ax.plot(sig_sorted["Phase"], sig_sorted["Normalized Current"], color=color, linewidth=1.8, label=sig_type)
        ax.set_title(cell_type)
        ax.set_xlim(0, 1)
        ax.set_ylim(-0.05, 1.05)
        if i % ncols == 0:
            ax.set_ylabel("Norm. Current")
        if i >= ncols * (nrows - 1):
            ax.set_xlabel("Phase")
        if i == 0:
            ax.legend(title="Signal type")
    for j in range(i+1, nrows*ncols):
        fig.delaxes(axes[j])
    suptitle = fig_title or "Averaged Waveforms by Cell Type"
    fig.suptitle(suptitle, fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
    return fig

plot_by_celltype(averaged_df)